In [ ]:
import sys
from pathlib import Path

# Garante que o pacote do projeto (src/spark_delta) seja encontrado mesmo que o
# kernel não tenha sido instalado com `uv sync` (fallback para src/).
try:
    import spark_delta  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
    sys.path.insert(0, str(Path.cwd().resolve() / "src"))

from spark_delta.config import settings
from spark_delta.spark_session import get_spark, WAREHOUSE_DIR
from spark_delta.storage import s3_client, ensure_bucket

s3 = s3_client()
for bucket in (settings.bronze_bucket, settings.warehouse_bucket):
    ensure_bucket(s3, bucket)

spark = get_spark("dml_delta")
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| warehouse:", WAREHOUSE_DIR)
spark

In [ ]:
from spark_delta.storage import list_objects

prefixos = s3.list_objects_v2(Bucket=settings.bronze_bucket, Delimiter="/").get("CommonPrefixes", [])
tabelas = sorted(p["Prefix"].rstrip("/") for p in prefixos)

for tabela in tabelas:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION 's3a://{settings.bronze_bucket}/{tabela}'
    """)

spark.sql("SHOW TABLES").show(truncate=False)

In [ ]:
# Versão atual de cada tabela que vamos alterar (para o Time Travel no final)
from delta.tables import DeltaTable

def caminho(tabela):
    return f"s3a://{settings.bronze_bucket}/{tabela}"

versao_inicial = {
    t: DeltaTable.forPath(spark, caminho(t)).history(1).select("version").first()[0]
    for t in ("produto", "estoque")
}
versao_inicial

In [ ]:
# Estado atual: categorias e os primeiros produtos com estoque
spark.sql("SELECT * FROM categoria ORDER BY cd_categoria").show()
spark.sql("""
    SELECT p.cd_produto, p.nm_produto, c.nm_categoria, p.preco, e.quantidade
    FROM produto p
    JOIN categoria c ON c.cd_categoria = p.cd_categoria
    JOIN estoque   e ON e.cd_produto   = p.cd_produto
    ORDER BY p.cd_produto
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Consulta reutilizada para acompanhar os 3 produtos do Trabalho 1
def produtos_trabalho1():
    spark.sql("""
        SELECT p.cd_produto, p.nm_produto, c.nm_categoria, p.preco,
               e.quantidade, e.estoque_minimo
        FROM produto p
        LEFT JOIN categoria c ON c.cd_categoria = p.cd_categoria
        LEFT JOIN estoque   e ON e.cd_produto   = p.cd_produto
        WHERE p.cd_produto BETWEEN 9001 AND 9003
        ORDER BY p.cd_produto
    """).show(truncate=False)

In [ ]:
spark.sql("SELECT COUNT(*) AS produtos_antes FROM produto").show()

spark.sql("""
    INSERT INTO produto
    SELECT 9001, 'Notebook Dell',    NULL, NULL, NULL, current_date(), current_timestamp() UNION ALL
    SELECT 9002, 'Mouse Logitech',   NULL, NULL, NULL, current_date(), current_timestamp() UNION ALL
    SELECT 9003, 'Teclado Mecânico', NULL, NULL, NULL, current_date(), current_timestamp()
""")

spark.sql("""
    INSERT INTO estoque
    SELECT 9001, 0, 5,  current_timestamp(), current_timestamp() UNION ALL
    SELECT 9002, 0, 10, current_timestamp(), current_timestamp() UNION ALL
    SELECT 9003, 0, 10, current_timestamp(), current_timestamp()
""")

spark.sql("SELECT COUNT(*) AS produtos_depois FROM produto").show()
produtos_trabalho1()

In [ ]:
# UPDATE via Spark SQL (produto)
spark.sql("UPDATE produto SET cd_categoria = 1, cd_fornecedor = 1, preco = 3500.00 WHERE cd_produto = 9001")
spark.sql("UPDATE produto SET cd_categoria = 2, cd_fornecedor = 2, preco = 120.00  WHERE cd_produto = 9002")
spark.sql("UPDATE produto SET cd_categoria = 2, cd_fornecedor = 2, preco = 250.00  WHERE cd_produto = 9003")

In [ ]:
# UPDATE via API Python DeltaTable (estoque) - mesmo efeito, sem SQL
from pyspark.sql import functions as F

dt_estoque = DeltaTable.forPath(spark, caminho("estoque"))
for cd_produto, qtd in [(9001, 15), (9002, 50), (9003, 30)]:
    dt_estoque.update(
        condition=F.col("cd_produto") == cd_produto,
        set={"quantidade": F.lit(qtd), "dt_atualizacao": F.current_timestamp()},
    )

produtos_trabalho1()

In [ ]:
spark.sql("DELETE FROM estoque WHERE cd_produto = 9003")      # via SQL
DeltaTable.forPath(spark, caminho("produto")).delete("cd_produto = 9003")  # via API DeltaTable

produtos_trabalho1()
spark.sql("SELECT COUNT(*) AS produtos_final FROM produto").show()

In [ ]:
from IPython.display import HTML, display
display(HTML("<style>pre { white-space: pre !important; }</style>"))

spark.sql("DESCRIBE HISTORY produto") \
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics") \
    .show(truncate=False)

In [ ]:
spark.sql("DESCRIBE HISTORY estoque").select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
v0 = versao_inicial["produto"]
v_atual = DeltaTable.forPath(spark, caminho("produto")).history(1).select("version").first()[0]

for v in range(v0, v_atual + 1):
    print(f"=== produto VERSION AS OF {v} ===")
    (spark.read.format("delta").option("versionAsOf", v).load(caminho("produto"))
        .where("cd_produto >= 9001")
        .select("cd_produto", "nm_produto", "cd_categoria", "preco")
        .orderBy("cd_produto")
        .show())

In [ ]:
# Diferença entre a versão anterior ao DML e a atual
antes = spark.read.format("delta").option("versionAsOf", v0).load(caminho("produto")).drop("_dt_ingestao")
depois = spark.read.format("delta").load(caminho("produto")).drop("_dt_ingestao")

print(f"Versão {v0}: {antes.count()} produtos | Atual (v{v_atual}): {depois.count()} produtos")
print("Linhas que existem só na versão atual:")
depois.subtract(antes).show(truncate=False)

In [ ]:
spark.stop()
print("SparkSession finalizada.")